# UrbanHub, Partie 3 : flux IoT pollution (OpenAQ)

**Objectif** : analyser la pollution de l'air dans les principales villes françaises à partir des capteurs OpenAQ,
ingérés en continu (toutes les 10 min) dans MinIO.

| Couche | Emplacement | Contenu |
|---|---|---|
| bronze | MinIO `urbanhub/bronze/openaq/pollution/date=…/` | réponses API brutes (JSONL gzip) |
| clean | `pollution.sensors`, `pollution.measurements` | mesures nettoyées, en µg/m³, UTC |
| analytique | vues `pollution.v_*` | une vue par question métier |

Polluants : PM2.5, PM10, NO2, O3, CO. Les seuils de référence (OMS 2021, seuils d'information et d'alerte FR/UE)
sont dans `pollution.thresholds`.

In [ ]:
import pandas as pd
import plotly.express as px
from scipy import stats
from urbanhub_common import read_sql

LABELS = {"pm25": "PM2.5", "pm10": "PM10", "no2": "NO2", "o3": "O3", "co": "CO"}
COLORS = {"pm25": "#2a78d6", "pm10": "#eb6834", "no2": "#1baf7a", "o3": "#eda100", "co": "#e87ba4"}
pd.set_option("display.float_format", "{:.2f}".format)

## 0. Vue d'ensemble et qualité des données

In [ ]:
fresh = read_sql("SELECT * FROM pollution.v_freshness ORDER BY city, pollutant")
print(f"{fresh.n_sensors.sum()} capteurs actifs, {fresh.n_measures.sum():,} mesures, "
      f"dernière mesure : {fresh.last_measure.max()}")
fresh.pivot(index="city", columns="pollutant", values="n_sensors").fillna(0).astype(int)

> 📖 **Comment lire ce graphique.** Nombre de capteurs actifs par ville et par polluant. Paris concentre le plus de capteurs (30 pour le NO2) ; le CO n'est mesuré que par 4 capteurs à Paris et 1 dans quelques villes, d'où une série très creuse.

In [ ]:
# Complétude : part des heures attendues effectivement reçues, par capteur
completeness = read_sql('''
    SELECT s.city, s.pollutant, s.sensor_id,
           count(m.*)::float / greatest(1, extract(epoch FROM max(m.measured_at) - min(m.measured_at)) / 3600 + 1) AS completeness
    FROM pollution.sensors s JOIN pollution.measurements m USING (sensor_id)
    GROUP BY 1, 2, 3''')
completeness.groupby("pollutant")["completeness"].describe()

> 📖 **Comment lire ce graphique.** Complétude = part des heures attendues effectivement reçues. Une médiane proche de 0,9 signifie que 90 % des heures sont présentes. Les PM10 sont plus lacunaires (médiane 0,49) : plusieurs capteurs ne transmettent que des moyennes journalières ou ont des interruptions.

In [ ]:
# Bilan du nettoyage (lignes reçues / insérées / rejetées par exécution)
runs = read_sql('''SELECT mode, count(*) AS runs, sum(api_calls) AS api_calls, sum(rows_received) AS recues,
                          sum(rows_inserted) AS inserees, sum(rows_rejected) AS rejetees
                   FROM ops.ingestion_runs WHERE part = 'pollution' AND status = 'success' GROUP BY mode''')
runs

> 📖 **Comment lire ce graphique.** Bilan des ingestions par mode. `recues` = lignes renvoyées par l'API, `inserees` = nouvelles lignes (les doublons sont ignorés), `rejetees` = lignes écartées par le nettoyage (valeurs négatives, périmées, aberrantes). Le flux (`stream`) insère peu, car OpenAQ ne publie qu'une valeur par heure : la plupart des relevés toutes les 10 min sont des doublons, ce qui est attendu.

## Q1. Quelles zones présentent les niveaux de pollution les plus élevés ?

Pour comparer des polluants entre eux, on rapporte chaque concentration moyenne au seuil OMS sur 24 h
(**ratio OMS** : au-dessus de 1, la recommandation est dépassée).

In [ ]:
city_lvl = read_sql("SELECT * FROM pollution.v_city_levels")
fig = px.bar(city_lvl, x="who_ratio", y="city", color="pollutant", barmode="group", orientation="h",
             color_discrete_map=COLORS, height=600, title="Concentration moyenne / seuil OMS, par ville",
             labels={"who_ratio": "ratio OMS", "city": "", "pollutant": "Polluant"})
fig.add_vline(x=1, line_dash="dot")
fig.show()
city_lvl.pivot(index="city", columns="pollutant", values="share_days_above_who").style.format("{:.0%}")

> 📖 **Comment lire ce graphique.** Barres = concentration moyenne divisée par le seuil OMS (ligne pointillée = 1). Le tableau donne la part des jours au-dessus du seuil OMS.

In [ ]:
zones = read_sql("SELECT * FROM pollution.v_zone_index ORDER BY zone_index DESC")
display(zones.head(15))
px.scatter_map(zones, lat="latitude", lon="longitude", color="zone_index", size="zone_index",
               hover_name="location_name", hover_data=["city", "worst_pollutant"],
               color_continuous_scale="Oranges", zoom=4.6, map_style="open-street-map", height=550,
               title="Indice de zone (pire ratio OMS parmi les polluants mesurés)").show()

> 📖 **Comment lire ce graphique.** Carte des sites de mesure : couleur et taille = **indice de zone** (pire ratio OMS parmi les polluants mesurés sur le site). Le tableau liste les 15 sites les plus chargés.

**Interprétation Q1.** Les zones les plus polluées sont toutes des **sites de trafic**, et toutes pour le **NO2** :
A1 à Saint-Denis (indice 2,0, soit 2 fois le seuil OMS), Marseille Plombières (1,9), périphérique de Lyon (1,8),
Montpellier Liberté (1,7), RN20 à Montlhéry (1,6), périphérique Est à Paris (1,4), A35 à Strasbourg (1,4).
À l'échelle des villes, **Marseille (1,07)** dépasse en moyenne le seuil OMS du NO2, et Strasbourg (0,98)
et Paris (0,93) s'en approchent. Les particules restent loin du seuil en automne (ratio 0,25 à 0,58).
**Conclusion** : la pollution est **très localisée le long des grands axes routiers**. Ce sont les riverains
de ces axes qu'il faut cibler en priorité, plutôt que la ville entière.

## Q2. Existe-t-il une variation journalière de la pollution ?

In [ ]:
prof = read_sql("SELECT * FROM pollution.v_hourly_profile")
for p in ["no2", "o3", "pm25"]:
    d = prof[(prof.pollutant == p) & prof.city.isin(["Paris", "Lyon", "Marseille", "Toulouse"])]
    px.line(d.sort_values("hour_local"), x="hour_local", y="mean_value", color="city", facet_col="day_type",
            markers=True, height=380, title=f"Profil journalier moyen : {LABELS[p]}",
            labels={"hour_local": "heure locale", "mean_value": "µg/m³"}).show()

> 📖 **Comment lire ce graphique.** Une figure par polluant : concentration moyenne pour chaque heure locale, en semaine (à gauche) et le week-end (à droite), pour 4 grandes villes.

In [ ]:
# Test statistique : la concentration dépend-elle de l'heure ? (Kruskal-Wallis, non paramétrique)
hourly = read_sql('''SELECT city, pollutant, extract(hour FROM hour_utc AT TIME ZONE 'Europe/Paris')::int AS h, value
                     FROM pollution.v_city_hourly''')
rows = []
for (city, p), g in hourly.groupby(["city", "pollutant"]):
    groups = [x.value.values for _, x in g.groupby("h") if len(x) > 5]
    if len(groups) >= 12:
        H, pval = stats.kruskal(*groups)
        rows.append({"city": city, "pollutant": p, "H": H, "p_value": pval,
                     "amplitude": g.groupby("h").value.mean().agg(lambda s: s.max() - s.min())})
pd.DataFrame(rows).sort_values(["pollutant", "H"], ascending=[True, False])

> 📖 **Comment lire ce graphique.** Test de **Kruskal-Wallis** : les concentrations diffèrent-elles selon l'heure ? H est d'autant plus grand que les heures se distinguent ; p < 0,05 indique une variation horaire significative. L'amplitude est l'écart en µg/m³ entre l'heure la plus polluée et la plus propre.

**Interprétation Q2 : oui, la variation journalière est forte et significative** (p < 0,001 pour toutes les
villes et tous les polluants).
- **NO2** : pic vers **10 h** en semaine (31 µg/m³ en moyenne, contre 16 µg/m³ le week-end à la même heure). C'est la
  signature du **trafic du matin**, renforcée par une atmosphère encore peu mélangée. Il remonte le soir (22-23 h),
  quand la couche d'air proche du sol s'abaisse et piège les émissions. L'amplitude atteint 26 µg/m³ à
  Strasbourg et à Marseille.
- **O3** : profil inverse, avec un minimum vers 9-10 h (34 µg/m³) et un **maximum vers 17 h** (85 µg/m³). Il se forme
  au soleil, et le NO2 du matin le détruit. Son amplitude est la plus forte de tous les polluants (42 à 62 µg/m³).
- **Particules** : variation plus faible (3 à 5 µg/m³ pour les PM2.5), car elles dépendent davantage des masses
  d'air que du trafic local.

## Q3. Quels polluants sont les plus dominants dans la ville étudiée ?

Pour chaque heure et chaque ville, le polluant dominant est celui dont le sous-indice (valeur / seuil OMS) est le plus élevé.

In [ ]:
dom = read_sql("SELECT * FROM pollution.v_dominant_pollutant")
px.bar(dom, x="share_dominant", y="city", color="pollutant", orientation="h", color_discrete_map=COLORS,
       category_orders={"pollutant": list(LABELS)}, height=480,
       title="Part des heures où chaque polluant domine").update_xaxes(tickformat=".0%").show()
dom.pivot(index="city", columns="pollutant", values="share_dominant").style.format("{:.0%}")

> 📖 **Comment lire ce graphique.** Pour chaque ville, la part des heures où chaque polluant est **dominant**, c'est-à-dire le plus élevé rapporté à son seuil OMS. On ne compare pas les µg/m³ bruts, qui avantageraient toujours l'O3 et le CO.

In [ ]:
# Zoom sur la ville étudiée
CITY = "Paris"
d = read_sql("SELECT hour_utc, pollutant, value FROM pollution.v_city_hourly WHERE city = %s", (CITY,))
px.box(d, x="pollutant", y="value", color="pollutant", color_discrete_map=COLORS, points=False,
       title=f"Distribution des concentrations horaires : {CITY}", labels={"value": "µg/m³"}).show()

> 📖 **Comment lire ce graphique.** Boîte à moustaches des concentrations horaires à Paris : la boîte contient la moitié centrale des valeurs, le trait la médiane, les moustaches l'étendue typique.

**Interprétation Q3.** Deux familles de villes se dessinent :
- **NO2 dominant** (trafic) : Strasbourg (68 % des heures), Marseille (63 %), Paris (57 %), Montpellier (55 %), Lyon (54 %) ;
- **O3 dominant** (ensoleillement, air de fond) : Bordeaux (57 %), Nice (54 %), Rennes (54 %), Lille (51 %),
  Nantes (49 %), Toulouse (45 %).

Pour une ville comme **Paris**, le polluant à surveiller est le **NO2** : la réduction du trafic routier y a le plus
d'effet. Dans les villes à dominante O3, l'enjeu est régional (précurseurs, chaleur estivale) plus que local.

## Q4. Peut-on identifier des épisodes de pollution anormale ?

On combine deux méthodes :
1. **Réglementaire** : dépassement des seuils d'information ou d'alerte (O3 180/240, NO2 200/400 en horaire ;
   PM10 50/80 en moyenne journalière) ou de la ligne directrice OMS.
2. **Statistique** : z-score de chaque mesure par rapport aux 7 jours précédents du même capteur
   (|z| > 3 avec au moins 48 h d'historique).

In [ ]:
epi = read_sql("SELECT * FROM pollution.v_pollution_episodes ORDER BY day")
display(epi.groupby(["episode_level", "pollutant"]).size().unstack(fill_value=0))
px.scatter(epi, x="day", y="city", color="episode_level", symbol="pollutant", size="who_ratio",
           height=450, title="Calendrier des épisodes (ville x jour)",
           color_discrete_map={"alerte": "#e34948", "information": "#eb6834",
                               "depassement_OMS": "#eda100", "anomalie_statistique": "#6250d6"}).show()

> 📖 **Comment lire ce graphique.** Tableau : nombre de jours × ville par niveau d'épisode et par polluant. Graphique : chaque point est un jour d'épisode. Sa couleur donne la gravité, sa forme le polluant, sa taille le ratio OMS.

In [ ]:
# Épisodes simultanés dans plusieurs villes : signe d'un phénomène régional (météo, transport de poussières...)
multi = epi[epi.episode_level != "anomalie_statistique"].groupby(["day", "pollutant"]).city.agg(["count", list])
multi[multi["count"] >= 3].sort_values("count", ascending=False)

> 📖 **Comment lire ce graphique.** Jours où **au moins 3 villes** sont en épisode pour le même polluant : on ne peut pas expliquer cela par une source locale. C'est le signe d'une situation **météorologique** commune (anticyclone, vent faible).

In [ ]:
# Série détaillée d'une ville avec les anomalies statistiques
CITY, P = "Paris", "pm10"
s = read_sql("SELECT hour_utc, value FROM pollution.v_city_hourly WHERE city=%s AND pollutant=%s ORDER BY 1", (CITY, P))
a = read_sql('''SELECT measured_at, value, z_score FROM pollution.v_anomalies
                WHERE city=%s AND pollutant=%s AND is_statistical_anomaly''', (CITY, P))
fig = px.line(s, x="hour_utc", y="value", title=f"{LABELS[P]} à {CITY} : anomalies (z > 3)",
              color_discrete_sequence=[COLORS[P]])
fig.add_scatter(x=a.measured_at, y=a.value, mode="markers", name="anomalie", marker=dict(color="#e34948", size=8, symbol="x"))
fig.show()

> 📖 **Comment lire ce graphique.** Série horaire d'un polluant dans une ville, avec des croix rouges sur les heures statistiquement anormales (z > 3 par rapport aux 7 jours précédents du même capteur).

**Interprétation Q4 : oui, on identifie des épisodes**, de natures différentes :
- **113 jours × ville de dépassement OMS pour le NO2** et 12 pour les PM2.5, entre fin août et octobre 2026 ;
- **des épisodes régionaux** : le NO2 dépasse le seuil OMS dans **7 villes en même temps les 24 et 25 septembre 2026**
  (et dans 5 villes les 15 et 23 septembre). Une telle simultanéité signe une situation météo stagnante, que la
  partie 4 confirme : vent faible, air qui ne se renouvelle pas ;
- **2 « alertes » NO2 à Lyon (7 et 8 octobre)** qui viennent d'**un seul capteur (LYON Centre, 400-500 µg/m³)**, quand
  le reste de la ville était autour de 25 µg/m³. C'est très probablement un **défaut de capteur**, ce qui montre
  l'intérêt de croiser seuils réglementaires et cohérence entre capteurs ;
- les anomalies statistiques (z > 3) sont nombreuses (155 jours pour les PM2.5) mais souvent modestes en valeur absolue : ce
  sont des hausses relatives par rapport à une semaine calme, pas forcément des niveaux dangereux.